# Jhadina Director — Google AI Plus Colab GPU and synthetic training canary
**Owner-run, never automatic.** This notebook evaluates optional interactive Colab capacity for the existing Director character-training pipeline. It does **not** train a real Director character or promote LoRA weights.

Google AI Plus includes a nominal **50 monthly compute units**, but remaining units, accelerator access and live rates are dynamic. Running a Colab GPU runtime can consume units *even when idle*. The notebook cannot read your subscription or enforce a spending cap. Check your actual Colab balance, usage rate and Runtime type first, and disconnect when finished.

**Safety:** The default below does not execute the canary. To train a tiny **synthetic-only** neural network, manually approve this individual trial and change RUN_SYNTHETIC_TRAINING to True. Only Python/Torch built-ins are used; no model downloads, private assets, secrets, external network calls, Drive mounts, or GPU provisioning APIs. Copy the resulting JSON manually if you want to retain it. Do not count this receipt as trained Director characters, real data provenance, GPU uptime, or production certification.

Notebook source: `infrastructure/homebase/google-drive/DIRECTOR-COLAB-AI-PLUS-PREFLIGHT.ipynb`.

In [ ]:
# Human-controlled one-shot admission, not an API quota meter.
ACCOUNT_PLAN_CONFIRMED = True
REPORTED_REMAINING_UNITS = None  # Enter the remaining units shown by your Colab UI.
PLANNED_EXPERIMENT_UNITS = 2    # Planning estimate only. NOT a hard enforced limit.
RESERVED_UNITS = 10
OWNER_APPROVES_THIS_TRIAL = False
RUN_SYNTHETIC_TRAINING = False  # Explicitly opt in only after reviewing quota.
assert ACCOUNT_PLAN_CONFIRMED, "Colab plan has not been confirmed"
assert REPORTED_REMAINING_UNITS is not None, "Enter observed remaining compute units"
assert isinstance(REPORTED_REMAINING_UNITS, (int, float)) and REPORTED_REMAINING_UNITS >= 0
assert isinstance(PLANNED_EXPERIMENT_UNITS, int) and 0 < PLANNED_EXPERIMENT_UNITS <= 10
assert isinstance(RESERVED_UNITS, int) and 0 <= RESERVED_UNITS <= 50
assert REPORTED_REMAINING_UNITS >= PLANNED_EXPERIMENT_UNITS + RESERVED_UNITS, "Trial would violate planned reserve"
assert OWNER_APPROVES_THIS_TRIAL, "Approval for this exact trial is required"
print("Manual capacity admission passed; this does not reserve or spend-limit Colab units.")

In [ ]:
import hashlib, json, platform, time
from pathlib import Path
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch not installed in this runtime; do not auto-install") from error

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU assigned. Change Runtime type manually if desired; no model training run.")
gpu = torch.cuda.get_device_properties(0)
receipt = {
    "schema": "jhadina.director.colab-ai-plus.synthetic-gpu-canary.v1",
    "compute_origin": "owner-interactive-colab",
    "gpu_name": str(gpu.name),
    "torch_version": torch.__version__,
    "python_version": platform.python_version(),
    "reported_remaining_units_before": float(REPORTED_REMAINING_UNITS),
    "planned_units_not_enforced": PLANNED_EXPERIMENT_UNITS,
    "reserve_units_not_enforced": RESERVED_UNITS,
    "gpu_available": True,
    "synthetic_gpu_optimizer_verified": False,
    "model_training_completed": False,
    "director_character_trained": False,
    "checkpoint_promotion_approved": False,
    "can_publish": False,
    "production_certified": False,
    "actual_provider_compute_units_consumed": None,
}
if RUN_SYNTHETIC_TRAINING:
    torch.manual_seed(7)
    device = torch.device("cuda:0")
    model = torch.nn.Linear(4, 2).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    source = torch.arange(32, dtype=torch.float32, device=device).reshape(8, 4) / 32
    target = torch.zeros((8, 2), device=device)
    started = time.monotonic()
    optimizer.zero_grad()
    loss = torch.nn.functional.mse_loss(model(source), target)
    loss.backward()
    optimizer.step()
    torch.cuda.synchronize()
    digest = hashlib.sha256(b"".join(p.detach().cpu().numpy().tobytes() for p in model.parameters())).hexdigest()
    receipt["synthetic_gpu_optimizer_verified"] = True
    receipt["synthetic_training_elapsed_seconds"] = round(time.monotonic() - started, 3)
    receipt["synthetic_weight_sha256"] = digest
print(json.dumps(receipt, indent=2))
# Optional local, ephemeral receipt only. No automatic Drive upload.
Path("/content/director_colab_synthetic_preflight.json").write_text(json.dumps(receipt, indent=2), encoding="utf-8")
print("Disconnect your Colab runtime to stop any remaining compute consumption.")